# Video Transcript Question Answering (RAG) — Hugging Face

This project uses LangChain, YouTube transcripts, FAISS, and Hugging Face to answer questions based on video content.

It extracts video transcripts, splits them into smaller chunks, and uses Hugging Face embeddings to retrieve relevant information. The Qwen2.5-1.5B-Instruct model generates answers based on the retrieved content.

## Install libraries

In [1]:
%pip -q install -U youtube-transcript-api langchain-community langchain-google-genai langchain-text-splitters faiss-cpu python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [35]:
!pip install -q -U transformers accelerate langchain-huggingface

In [6]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

## Step 1a - Indexing (Document Ingestion)

In [11]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
try:
    fetched = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched.to_raw_data()
    transcript = " ".join(chunk['text'] for chunk in transcript_list)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [12]:
if not transcript.strip():
    raise ValueError('This video transcript is empty.')
print(f"Transcript ready ({len(transcript):,} characters). Preview:")
print(transcript[:700])

Transcript ready (133,836 characters). Preview:
the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we


In [13]:
transcript_list[:2]  # preview the first two transcript segments

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96}]

## Step 1b - Indexing (Text Splitting)

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [15]:
len(chunks)

168

In [16]:
chunks[min(100, len(chunks) - 1)]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [20]:
!pip install -q langchain-huggingface sentence-transformers

In [18]:
embedding = GoogleGenerativeAIEmbeddings(model='gemini-embedding-2')


In [21]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
vector_store.index_to_docstore_id

{0: 'b20fe2c4-5c2b-4892-be7c-fbf3b0015ff2',
 1: 'c49fd550-7d31-47e4-b25f-0bb07dff6a2f',
 2: 'fbc15bfa-1446-4871-869c-1f0d56e29276',
 3: 'cf6cf189-48f9-4fc6-8404-4be6c6d12e1f',
 4: '0ef2dd66-7733-41af-9c06-2353fa1bb24f',
 5: 'edddb4a6-a335-4079-9d19-f39c76d9581f',
 6: '2b7cd1d0-c110-4142-9424-65cb687766ad',
 7: '483c86d8-3b58-4a83-b065-b5cdaf9ebcf3',
 8: '1ce1dc9b-91f9-4bf8-8a62-766069ceebc9',
 9: '336edf08-13e5-4621-8534-f186b639de37',
 10: 'c873f93b-5904-42d5-8961-35240cbed1eb',
 11: 'd39cf16c-f1a5-4718-ba10-eb7c504f24f9',
 12: '783f082f-63c9-4f19-93b6-32b29a11a8db',
 13: '21a5713a-a179-4724-9fd6-ba5e73d5cc32',
 14: 'c10b7053-5c79-44bf-b2c6-f62539b0b985',
 15: 'a4ed47b3-c4a2-4020-899e-a6224e66643f',
 16: 'a0757de4-58ca-4261-909f-361a99a91927',
 17: 'd8666b65-d8ee-4cc2-9e1b-5966ab95d334',
 18: 'e44e0b27-d311-415f-b356-6dc05564c560',
 19: '9a089458-6bc6-4f93-8e96-57336e959d29',
 20: 'e555e83d-818b-467c-8d14-b49ffe2e31f6',
 21: 'fc5bc176-ff76-43b6-99fa-d4822deeb0e6',
 22: '7cdc0d39-0776-

In [23]:
first_doc_id = next(iter(vector_store.index_to_docstore_id.values()))
vector_store.get_by_ids([first_doc_id])

[Document(id='b20fe2c4-5c2b-4892-be7c-fbf3b0015ff2', metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal qu

## Step 2 - Retrieval

In [24]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [25]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7c05ccb0ba10>, search_kwargs={'k': 4})

In [26]:
retriever.invoke('What is deepmind')

[Document(id='2e035eb4-e682-4dae-bc35-8bda1b6497a3', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

## Step 3 - Augmentation

In [36]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    dtype="auto",
    max_new_tokens=200,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=generator)
model = ChatHuggingFace(llm=llm)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [37]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [38]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [39]:
retrieved_docs

[Document(id='32d8c386-d19c-402d-803d-5c945b7f0587', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [40]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [41]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [42]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we 

## Step 4 - Generation

In [43]:
answer = model.invoke(final_prompt)
print(answer.content)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user

      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we yo